In [7]:
# POST /ai/query — payloads desde `ai_query_test_payloads.ipynb` + llamadas al API
# Requisito: `uvicorn app.main:app --reload` en marcha.
import json
import re
from pathlib import Path

import httpx

BASE_URL = "http://127.0.0.1:8000"
QUERY_PATH = "/ai/query"
TIMEOUT = httpx.Timeout(120.0)
AUTH_HEADERS = {
    "Authorization": "Bearer dev-token",
    "Content-Type": "application/json",
}


def _resolve_payloads_path() -> Path:
    here = Path.cwd().resolve()
    for p in (
        here / "notebooks" / "ai_query_test_payloads.ipynb",
        here / "ai_query_test_payloads.ipynb",
        here.parent / "notebooks" / "ai_query_test_payloads.ipynb",
    ):
        if p.is_file():
            return p
    raise FileNotFoundError(
        "No encontré ai_query_test_payloads.ipynb. Abre el kernel con cwd = raíz del repo o notebooks/."
    )


def load_payload_dict_from_payloads_notebook(ipynb: Path) -> dict[str, dict]:
    """Clave = línea `## ...` de la celda markdown; valor = JSON del fence ```json."""
    nb = json.loads(ipynb.read_text(encoding="utf-8"))
    result: dict[str, dict] = {}
    for cell in nb.get("cells", []):
        if cell.get("cell_type") != "markdown":
            continue
        source = "".join(cell.get("source", []))
        heading: str | None = None
        for line in source.splitlines():
            stripped = line.strip()
            if stripped.startswith("## ") and not stripped.startswith("###"):
                heading = stripped[3:].strip()
        for m in re.finditer(
            r"```json\s*(?:\r?\n)([\s\S]*?)(?:\r?\n)```",
            source,
        ):
            if not heading:
                continue
            payload = json.loads(m.group(1).strip())
            key = heading
            n = 2
            while key in result:
                key = f"{heading} ({n})"
                n += 1
            result[key] = payload
    return result


def _response_text_from_sse(body: bytes) -> str | None:
    """Reconstruye el texto del asistente desde eventos `token` (stream SSE)."""
    try:
        text = body.decode("utf-8")
    except UnicodeDecodeError:
        return None
    parts: list[str] = []
    for block in text.split("\n\n"):
        block = block.strip()
        if not block:
            continue
        event_name: str | None = None
        data_str: str | None = None
        for line in block.split("\n"):
            if line.startswith("event:"):
                event_name = line[6:].strip()
            elif line.startswith("data:"):
                data_str = line[5:].strip()
        if event_name == "error":
            return None
        if event_name == "token" and data_str:
            try:
                obj = json.loads(data_str)
                parts.append(obj.get("text") or "")
            except json.JSONDecodeError:
                continue
    out = "".join(parts)
    return out if out else None


def fetch_response_text(client: httpx.Client, payload: dict) -> str | None:
    url = f"{BASE_URL}{QUERY_PATH}"
    try:
        if payload.get("stream"):
            with client.stream(
                "POST",
                url,
                json=payload,
                headers=AUTH_HEADERS,
                timeout=TIMEOUT,
            ) as r:
                if r.status_code >= 400:
                    return None
                body = r.read()
            return _response_text_from_sse(body)
        r = client.post(url, json=payload, headers=AUTH_HEADERS, timeout=TIMEOUT)
        if r.status_code >= 400:
            return None
        data = r.json()
        return data.get("response_text")
    except (httpx.HTTPError, json.JSONDecodeError, ValueError):
        return None

In [8]:
PAYLOADS: dict[str, dict] = load_payload_dict_from_payloads_notebook(
    _resolve_payloads_path()
)
print(f"{len(PAYLOADS)} payloads")
list(PAYLOADS.keys())

38 payloads


['1 · Chatbot básico — pregunta general',
 '2 · Chatbot con filtro de país',
 '3 · Chatbot con framework_filter único',
 '4 · Chatbot con múltiples frameworks',
 '5 · Chatbot con response_mode structured',
 '6 · Chatbot con streaming',
 '7 · Chatbot con project_id (memoria L2)',
 '8 · Chatbot continuando una conversación (thread_id)',
 '9 · Chatbot con project_id + thread_id (L1 + L2)',
 '10 · Chatbot con top_k alto',
 '11 · Chatbot con top_k mínimo',
 '12 · Chatbot sin biblioteca global',
 '13 · Chatbot — query de seguimiento ("¿por qué?")',
 '14 · Chatbot — pregunta fuera de dominio (guardrail)',
 '15 · Chatbot — intento de prompt injection',
 '16 · Chatbot — query en inglés',
 '17 · Chatbot — query muy corta',
 '18 · Chatbot — sin documentos relevantes en RAG',
 '19 · Executive Summary básico',
 '20 · Executive Summary con filtro de framework',
 '21 · Policy Generation básico',
 '22 · Policy Generation con project_id',
 '23 · Policy Generation con framework_filter',
 '24 · Policy Ge

In [9]:
RESPONSE_TEXTS: dict[str, str | None] = {}

with httpx.Client(timeout=TIMEOUT) as client:
    for title, body in PAYLOADS.items():
        RESPONSE_TEXTS[title] = fetch_response_text(client, body)

ok = sum(1 for v in RESPONSE_TEXTS.values() if v is not None)
print(f"Respuestas con texto: {ok} / {len(RESPONSE_TEXTS)}")
RESPONSE_TEXTS

Respuestas con texto: 24 / 38


{'1 · Chatbot básico — pregunta general': 'El documento ISO 42001 menciona que la organización debe establecer una estructura de gobernanza adecuada para gestionar los sistemas de IA, lo que incluye definir roles y responsabilidades claras. Sin embargo, no se menciona específicamente un "comité de gobernanza de IA". En cambio, se enfatiza la importancia de la alta dirección y la creación de políticas que alineen la gestión de IA con los objetivos organizacionales [Fuente: ISO42001_Replica.pdf, Anexo D]. \n\nSi necesitas más detalles sobre la estructura de gobernanza o cómo se implementa, házmelo saber.',
 '2 · Chatbot con filtro de país': 'No encontré información específica sobre los requisitos legales para usar IA en procesos de crédito en los documentos recuperados. Sin embargo, es importante considerar que el uso de IA en este contexto puede estar sujeto a regulaciones relacionadas con la protección de datos, la no discriminación y la transparencia en la toma de decisiones. \n\nTe r

In [11]:
PAYLOADS

{'1 · Chatbot básico — pregunta general': {'query': '¿Qué dice ISO 42001 sobre el comité de gobernanza de IA?',
  'use_case': 'chatbot'},
 '2 · Chatbot con filtro de país': {'query': '¿Cuáles son los requisitos legales para usar IA en procesos de crédito?',
  'use_case': 'chatbot',
  'country': 'costa-rica'},
 '3 · Chatbot con framework_filter único': {'query': '¿Cuáles son los controles del Anexo A de ISO 42001?',
  'use_case': 'chatbot',
  'framework_filter': ['iso_42001']},
 '4 · Chatbot con múltiples frameworks': {'query': '¿Cómo se relacionan los controles de NIST AI RMF con los requisitos de ISO 42001?',
  'use_case': 'chatbot',
  'framework_filter': ['iso_42001', 'nist_airmf'],
  'top_k': 10},
 '5 · Chatbot con response_mode structured': {'query': '¿Qué obligaciones tiene la organización respecto a la transparencia algorítmica?',
  'use_case': 'chatbot',
  'response_mode': 'structured'},
 '6 · Chatbot con streaming': {'query': 'Explica qué es un sistema de IA de alto riesgo segú

In [13]:
COMBINED: dict[str, dict] = {
    key: {
        "title": key,
        "payload": PAYLOADS[key],
        "response_text": RESPONSE_TEXTS.get(key),
    }
    for key in PAYLOADS
}

COMBINED
# COMBINED["1 · Chatbot básico — pregunta general"]["response_text"]

{'1 · Chatbot básico — pregunta general': {'title': '1 · Chatbot básico — pregunta general',
  'payload': {'query': '¿Qué dice ISO 42001 sobre el comité de gobernanza de IA?',
   'use_case': 'chatbot'},
  'response_text': 'El documento ISO 42001 menciona que la organización debe establecer una estructura de gobernanza adecuada para gestionar los sistemas de IA, lo que incluye definir roles y responsabilidades claras. Sin embargo, no se menciona específicamente un "comité de gobernanza de IA". En cambio, se enfatiza la importancia de la alta dirección y la creación de políticas que alineen la gestión de IA con los objetivos organizacionales [Fuente: ISO42001_Replica.pdf, Anexo D]. \n\nSi necesitas más detalles sobre la estructura de gobernanza o cómo se implementa, házmelo saber.'},
 '2 · Chatbot con filtro de país': {'title': '2 · Chatbot con filtro de país',
  'payload': {'query': '¿Cuáles son los requisitos legales para usar IA en procesos de crédito?',
   'use_case': 'chatbot',
   '